In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

import torch
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

config = {
    "train_file": "pretraining_augmented_data/train.jsonl",
    "eval_file": "pretraining_augmented_data/eval.jsonl",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,  # Can be higher with 3B
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 2,  # Can be lower
    "learning_rate": 2e-5,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 512,  # Can be longer
    "output_dir": "medical_qwen_3b_cpt",
    "seed": 42,
}

print("Qwen2.5-3B Full Training Config")
print(f"Batch size: {config['per_device_train_batch_size']}")
print(f"Accumulation: {config['gradient_accumulation_steps']}")

Qwen2.5-3B Full Training Config
Batch size: 2
Accumulation: 2


In [2]:
def load_jsonl(file_path: str) -> List[Dict]:
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except:
                pass
    return data

train_data = load_jsonl(config["train_file"])
eval_data = load_jsonl(config["eval_file"])

print(f" Train: {len(train_data):,} chunks")
print(f" Eval: {len(eval_data):,} chunks")

 Train: 7,540 chunks
 Eval: 16 chunks


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen2.5-3B",
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(" Tokenizer loaded")

 Tokenizer loaded


In [4]:
from transformers import AutoModelForCausalLM

print("Loading Qwen2.5-3B in bfloat16...")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = AutoModelForCausalLM.from_pretrained(
    "Qwen2.5-3B",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model.gradient_checkpointing_enable()

# Verify gradients enabled
for param in model.parameters():
    param.requires_grad = True

print(f" Model loaded: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params")
allocated = torch.cuda.memory_allocated(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"   GPU: {allocated:.2f} / {total:.2f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen2.5-3B in bfloat16...


Loading weights: 100%|███████████████████████████████████████████████████████████████| 434/434 [00:00<00:00, 639.91it/s]


 Model loaded: 3.09B params
   GPU: 6.17 / 34.19 GB


In [5]:
from datasets import Dataset
from torch.utils.data import DataLoader

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=config["max_seq_length"],
        padding="max_length",
    )

train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=100, remove_columns=["text"])

eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})
eval_dataset = eval_dataset.map(tokenize_function, batched=True, batch_size=100, remove_columns=["text"])

def collate_fn(batch):
    return {
        'input_ids': torch.stack([torch.tensor(x['input_ids']) for x in batch]),
        'attention_mask': torch.stack([torch.tensor(x['attention_mask']) for x in batch]),
    }

train_loader = DataLoader(train_dataset, batch_size=config["per_device_train_batch_size"], shuffle=True, collate_fn=collate_fn, pin_memory=False, num_workers=0)
eval_loader = DataLoader(eval_dataset, batch_size=config["per_device_eval_batch_size"], shuffle=False, collate_fn=collate_fn, pin_memory=False, num_workers=0)

print(f"✅ Train: {len(train_dataset):,} samples")
print(f"✅ Eval: {len(eval_dataset):,} samples")

Map: 100%|█████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 1893.80 examples/s]

✅ Train: 7,540 samples
✅ Eval: 16 samples


In [6]:
from bitsandbytes.optim import AdamW8bit
from torch.optim.lr_scheduler import LinearLR

optimizer = AdamW8bit(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)

total_steps = (len(train_loader) * config["num_train_epochs"]) // config["gradient_accumulation_steps"]
scheduler = LinearLR(optimizer, start_factor=1.0, total_iters=total_steps)

print(f"✅ Optimizer: AdamW8bit")
print(f"✅ Total steps: {total_steps}")

✅ Optimizer: AdamW8bit
✅ Total steps: 5655


In [7]:
output_dir = Path(config["output_dir"])
output_dir.mkdir(exist_ok=True)

model.train()
global_step = 0
best_eval_loss = float('inf')
patience = 5
patience_counter = 0

print("\n" + "="*80)
print("🚀 STARTING QWEN 3B TRAINING")
print("="*80 + "\n")

for epoch in range(config["num_train_epochs"]):
    print(f"Epoch {epoch + 1}/{config['num_train_epochs']}")
    epoch_loss = 0
    step_count = 0
    
    for batch_idx, batch in enumerate(train_loader):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['input_ids'],
        )
        
        loss = outputs.loss / config["gradient_accumulation_steps"]
        loss.backward()
        epoch_loss += loss.item() * config["gradient_accumulation_steps"]
        step_count += 1
        
        if (batch_idx + 1) % config["gradient_accumulation_steps"] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            
            if global_step % 10 == 0:  # Log every 10 steps (faster)
                print(f"Step {global_step}: loss = {epoch_loss/step_count:.4f}")
            
            if global_step % 500 == 0:  # Eval less frequently
                model.eval()
                eval_loss = 0
                eval_count = 0
                
                with torch.no_grad():
                    for eval_batch in eval_loader:
                        eval_batch = {k: v.to(model.device) for k, v in eval_batch.items()}
                        eval_outputs = model(input_ids=eval_batch['input_ids'], attention_mask=eval_batch['attention_mask'], labels=eval_batch['input_ids'])
                        eval_loss += eval_outputs.loss.item()
                        eval_count += 1
                
                avg_eval_loss = eval_loss / eval_count
                print(f"  Eval loss: {avg_eval_loss:.4f}")
                
                # Early stopping logic
                if avg_eval_loss < best_eval_loss:
                    best_eval_loss = avg_eval_loss
                    patience_counter = 0
                    
                    # Save best model (safe)
                    try:
                        best_dir = output_dir / "best_model"
                        best_dir.mkdir(exist_ok=True)
                        model.save_pretrained(str(best_dir))
                        tokenizer.save_pretrained(str(best_dir))
                        print(f"  ✓ Best model saved (loss: {avg_eval_loss:.4f})")
                    except Exception as e:
                        print(f"  ⚠️  Save failed: {e}")
                else:
                    patience_counter += 1
                    print(f"  No improvement. Patience: {patience_counter}/{patience}")
                    
                    if patience_counter >= patience:
                        print(f"\n✅ EARLY STOPPING - No improvement for {patience} evals")
                        model.train()
                        break
                
                model.train()
                torch.cuda.empty_cache()
    
    # Break outer loop if early stopped
    if patience_counter >= patience:
        break

print("\n" + "="*80)
print("✅ TRAINING COMPLETE")
print("="*80)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.



🚀 STARTING QWEN 3B TRAINING

Epoch 1/3
Step 10: loss = 2.0376
Step 20: loss = 1.8467
Step 30: loss = 1.8214
Step 40: loss = 1.7634
Step 50: loss = 1.7491
Step 60: loss = 1.7465
Step 70: loss = 1.7134
Step 80: loss = 1.7089
Step 90: loss = 1.6974
Step 100: loss = 1.6690
Step 110: loss = 1.6593
Step 120: loss = 1.6482
Step 130: loss = 1.6346
Step 140: loss = 1.6285
Step 150: loss = 1.6136
Step 160: loss = 1.5998
Step 170: loss = 1.5936
Step 180: loss = 1.5834
Step 190: loss = 1.5760
Step 200: loss = 1.5719
Step 210: loss = 1.5683
Step 220: loss = 1.5649
Step 230: loss = 1.5639
Step 240: loss = 1.5620
Step 250: loss = 1.5497
Step 260: loss = 1.5473
Step 270: loss = 1.5441
Step 280: loss = 1.5403
Step 290: loss = 1.5313
Step 300: loss = 1.5265
Step 310: loss = 1.5205
Step 320: loss = 1.5158
Step 330: loss = 1.5132
Step 340: loss = 1.5110
Step 350: loss = 1.5073
Step 360: loss = 1.5072
Step 370: loss = 1.5020
Step 380: loss = 1.4967
Step 390: loss = 1.4959
Step 400: loss = 1.4950
Step 410:

Writing model shards: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:08<00:00,  8.05s/it]


  ✓ Best model saved (loss: 2.2283)
Step 510: loss = 1.4815
Step 520: loss = 1.4802
Step 530: loss = 1.4808
Step 540: loss = 1.4780
Step 550: loss = 1.4757
Step 560: loss = 1.4750
Step 570: loss = 1.4720
Step 580: loss = 1.4681
Step 590: loss = 1.4654
Step 600: loss = 1.4664
Step 610: loss = 1.4642
Step 620: loss = 1.4618
Step 630: loss = 1.4607
Step 640: loss = 1.4576
Step 650: loss = 1.4559
Step 660: loss = 1.4553
Step 670: loss = 1.4534
Step 680: loss = 1.4520
Step 690: loss = 1.4512
Step 700: loss = 1.4507
Step 710: loss = 1.4502
Step 720: loss = 1.4490
Step 730: loss = 1.4465
Step 740: loss = 1.4440
Step 750: loss = 1.4416
Step 760: loss = 1.4414
Step 770: loss = 1.4408
Step 780: loss = 1.4398
Step 790: loss = 1.4386
Step 800: loss = 1.4372
Step 810: loss = 1.4356
Step 820: loss = 1.4352
Step 830: loss = 1.4331
Step 840: loss = 1.4340
Step 850: loss = 1.4325
Step 860: loss = 1.4325
Step 870: loss = 1.4319
Step 880: loss = 1.4290
Step 890: loss = 1.4281
Step 900: loss = 1.4258
Step

In [8]:
best_model_path = output_dir / "best_model"
best_model_path.mkdir(exist_ok=True)

model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

print(f"✅ Model saved to {best_model_path}")

Writing model shards: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:13<00:00, 13.29s/it]

✅ Model saved to medical_qwen_3b_cpt/best_model


In [9]:
import os
from pathlib import Path

model_path = Path(config["output_dir"]) / "best_model"

if model_path.exists():
    # Get all files in the model directory
    print("📦 Model Directory Contents:\n")
    
    total_size = 0
    for root, dirs, files in os.walk(model_path):
        for file in files:
            file_path = Path(root) / file
            size_bytes = file_path.stat().st_size
            total_size += size_bytes
            
            size_gb = size_bytes / (1024**3)
            size_mb = size_bytes / (1024**2)
            
            rel_path = file_path.relative_to(model_path)
            if size_gb > 0.001:
                print(f"  📄 {rel_path}: {size_gb:.3f} GB")
            else:
                print(f"  📄 {rel_path}: {size_mb:.2f} MB")
    
    total_gb = total_size / (1024**3)
    print(f"\n📊 Total Model Size: {total_gb:.2f} GB")
    print(f"✅ Model saved at: {model_path}")
else:
    print(f"❌ Model not found at: {model_path}")

📦 Model Directory Contents:

  📄 tokenizer_config.json: 0.00 MB
  📄 tokenizer.json: 0.011 GB
  📄 chat_template.jinja: 0.00 MB
  📄 generation_config.json: 0.00 MB
  📄 model.safetensors: 5.748 GB
  📄 config.json: 0.00 MB

📊 Total Model Size: 5.76 GB
✅ Model saved at: medical_qwen_3b_cpt/best_model


In [10]:
from transformers import AutoModelForCausalLM

# Load the model
model = AutoModelForCausalLM.from_pretrained(str(Path(config["output_dir"]) / "best_model"))
print(f"✅ Model loaded successfully")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

Loading weights: 100%|██████████████████████████████████████████████████████████████| 434/434 [00:00<00:00, 5801.42it/s]

✅ Model loaded successfully
Parameters: 3.09B


**Testing**

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = Path(config["output_dir"]) / "best_model"
model = AutoModelForCausalLM.from_pretrained(str(model_path), device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(str(model_path))

# Test with a textbook question
question = "What is chiva shunt type 1?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=200, temperature=0.7, top_p=0.9)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

Loading weights: 100%|███████████████████████████████████████████████████████████████| 434/434 [00:02<00:00, 190.87it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is chiva shunt type 1? A: Reflux from the deep to the superficial vein B: Reflux from the superficial to the deep vein C: Reflux from the deep vein to the perforating vein D: Reflux from the superficial vein to the deep vein
The correct answer is D: Reflux from the superficial vein to the deep vein. This type of shunt is characterized by blood flowing from a superficial vein into a communicating vein (usually a perforating vein) and then back into a deep vein, creating a circuit of abnormal flow.


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

model_path = Path("medical_qwen_3b_cpt") / "best_model"
model = AutoModelForCausalLM.from_pretrained(str(model_path), device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(str(model_path))

# Test with a textbook question
question = "What is chiva shunt type 1?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=200, temperature=0.7, top_p=0.9)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

Loading weights: 100%|███████████████████████████████████████████████████████████████| 434/434 [00:03<00:00, 144.16it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is chiva shunt type 1? A: Reflux from the deep to the superficial vein B: Reflux from the superficial to the deep vein C: Reflux from the deep vein to the perforating vein D: Reflux from the superficial vein to the deep vein
The correct answer is D: Reflux from the superficial vein to the deep vein. This type of shunt is characterized by blood flowing from a superficial vein into a communicating vein (usually a perforating vein) and then back into a deep vein, creating a circuit of abnormal flow.


In [12]:
question = "What is chiva shunt type 1?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=150, temperature=0.7)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("\nModel Answer:")
print(answer)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is chiva shunt type 1?

Model Answer:
What is chiva shunt type 1? A: Reflux from the deep to the superficial vein B: Reflux from the superficial to the deep vein C: Reflux from the deep vein to the perforating vein D: Reflux from the superficial vein to the deep vein
The correct answer is D: Reflux from the superficial vein to the deep vein. This type of shunt is characterized by blood flowing from a superficial vein into a communicating vein (usually a perforating vein) and then back into a deep vein, creating a circuit of abnormal flow.


In [18]:
question = "What is CHIVA?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=150, temperature=0)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("\nModel Answer:")
print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is CHIVA?

Model Answer:
What is CHIVA? CHIVA is a non-surgical method of treating varicose veins. It is a French acronym for "cure conservatrice et hemodynamique des veines ambulatoires" which translates to "conservative and hemodynamic treatment of ambulatory veins." CHIVA is based on the principle of restoring the normal flow of blood in the veins, rather than simply removing the varicose veins. This is achieved by using a combination of compression, massage, and other techniques to improve the function of the valves in the veins. CHIVA has been shown to be effective in treating varicose veins, and is now widely used in France and other countries.


In [14]:
import json

# Check 10 random entries
with open('pretraining_augmented_data/train.jsonl', 'r') as f:
    lines = f.readlines()
    
# Check if any contain MCQ format
mcq_count = 0
total = 0
for line in lines[:100]:  # Check first 100
    data = json.loads(line)
    text = data.get('text', '')
    if 'A:' in text and 'B:' in text and 'C:' in text and 'D:' in text:
        mcq_count += 1
        print(f"MCQ found:\n{text[:200]}...\n")
    total += 1

print(f"\nMCQ entries: {mcq_count}/{total}")


MCQ entries: 0/100


In [15]:
import re

def extract_answer_only(full_response):
    # Extract just the "correct answer" part
    match = re.search(r'The correct answer is (.+?)(?:\n|$)', full_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # Fallback: extract after "Answer:"
    match = re.search(r'Answer:\s*(.+?)(?:\n\n|$)', full_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    return full_response

question = "What is chiva shunt type 1?"
inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=200, temperature=0.7)
full_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract clean answer
clean_answer = extract_answer_only(full_answer)
print(f"Q: {question}\nA: {clean_answer}")

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is chiva shunt type 1?
A: D: Reflux from the superficial vein to the deep vein. This type of shunt is characterized by blood flowing from a superficial vein into a communicating vein (usually a perforating vein) and then back into a deep vein, creating a circuit of abnormal flow.


In [6]:
import re

def extract_answer_only(full_response):
    # Extract just the "correct answer" part
    match = re.search(r'The correct answer is (.+?)(?:\n|$)', full_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # Fallback: extract after "Answer:"
    match = re.search(r'Answer:\s*(.+?)(?:\n\n|$)', full_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    return full_response

question = "What is chiva shunt type 3?"
inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=200, temperature=0.7)
full_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract clean answer
clean_answer = extract_answer_only(full_answer)
print(f"Q: {question}\nA: {clean_answer}")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is chiva shunt type 3?
A: What is chiva shunt type 3? I have a shunt type 3 with a shunt 3.1. I have been told that I have a shunt type 3. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have been told that I have a shunt type 3.1. I have 

In [20]:
question = "Can you explain CHIVA Shunt Type 3?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=150, temperature=0)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("\nModel Answer:")
print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Can you explain CHIVA Shunt Type 3?

Model Answer:
Can you explain CHIVA Shunt Type 3? CHIVA is a French acronym that stands for "Contrôle et Harmonisation de la Circulation Veineuse des Appareils." It is a non-surgical treatment method for venous insufficiency, which is a condition where the veins in the body do not function properly, leading to the pooling of blood and the development of varicose veins.

Shunt Type 3 is one of the three types of venous insufficiency identified by the CHIVA method. In this type, blood flows from the deep veins (the veins that carry blood away from the heart) into the superficial veins (the veins that carry blood towards the heart) through a perforating vein. This means that the blood is bypassing the normal flow from the superficial veins to the deep veins, causing the superficial veins to become overloaded and leading to varicose veins.

The goal of CHIVA treatment for Shunt Type 3 is to redirect the blood flow from the superficial veins ba

In [7]:
question = "Can you explain CHIVA Shunt Type 1+2?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=150, temperature=0)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("\nModel Answer:")
print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Can you explain CHIVA Shunt Type 1+2?

Model Answer:
Can you explain CHIVA Shunt Type 1+2? CHIVA is a method of treating varicose veins. It is a non-surgical method of treating varicose veins. It is a method of treating varicose veins that is non-invasive. It is a method of treating varicose veins that is non-surgical. It is a method of treating varicose veins that is non-invasive and non-surgical. It is a method of treating varicose veins that is non-invasive, non-surgical, and non-thermal. It is a method of treating varicose veins that is non-invasive, non-surgical, non-thermal, and non-pharmaceutical. It is a method of treating varicose veins that is non-invasive, non-surgical, non-thermal, non-pharmaceutical, and non-chemical. It is a method of treating varicose veins that is non-invasive, non-surgical, non-thermal, non-pharmaceutical, non-chemical, and non-hormonal. It is a method of treating varicose veins that is non-invasive, non-surgical, non-thermal, non-pharmaceuti

In [51]:
question = "Can you explain CHIVA Shunt Type 2A?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=150, temperature=0)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("\nModel Answer:")
print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Can you explain CHIVA Shunt Type 2A?

Model Answer:
Can you explain CHIVA Shunt Type 2A? CHIVA is a method of treating varicose veins. It is a non-surgical method of treating varicose veins. It is a method of treating varicose veins that is non-invasive. It is a method of treating varicose veins that is non-surgical. It is a method of treating varicose veins that is non-invasive and non-surgical. It is a method of treating varicose veins that is non-invasive, non-surgical, and non-thermal. It is a method of treating varicose veins that is non-invasive, non-surgical, non-thermal, and non-pharmaceutical. It is a method of treating varicose veins that is non-invasive, non-surgical, non-thermal, non-pharmaceutical, and non-chemical. It is a method of treating varicose veins that is non-invasive, non-surgical, non-thermal, non-pharmaceutical, non-chemical, and non-hormonal. It is a method of treating varicose veins that is non-invasive, non-surgical, non-thermal, non-pharmaceutica

In [52]:
question = "Can you explain CHIVA Shunt Type 2B?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=150, temperature=0)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("\nModel Answer:")
print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Can you explain CHIVA Shunt Type 2B?

Model Answer:
Can you explain CHIVA Shunt Type 2B? CHIVA is a method of treating varicose veins. It is a non-surgical method of treating varicose veins. It is a method of treating varicose veins that is based on the idea that the veins that are causing the problem are not the problem. The problem is the valves in the veins that are not working properly. The CHIVA method is based on the idea that the veins that are causing the problem are not the problem. The problem is the valves in the veins that are not working properly. The CHIVA method is based on the idea that the veins that are causing the problem are not the problem. The problem is the valves in the veins that are not working properly. The CHIVA method is based on the idea that the veins that are causing the problem are not the problem. The problem is the valves in the veins that are not working properly. The CHIVA method is based on the idea that the veins that are causing the pro

In [21]:
question = "What is CHIVA Shunt Type 3?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=150, temperature=0)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("\nModel Answer:")
print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is CHIVA Shunt Type 3?

Model Answer:
What is CHIVA Shunt Type 3? CHIVA is a method of treating varicose veins without surgery. It is based on the principle of restoring the natural drainage of the leg by eliminating the refluxes. The CHIVA method is based on the study of the varicose veins by means of duplex ultrasound. The study identifies the varicose veins and the shunts that feed them. A shunt is a circuit that carries the blood from the point of entry to the point of exit. The CHIVA method is based on the elimination of the shunts that are responsible for the varicose veins. The CHIVA method is based on the elimination of the shunts that are responsible for the varicose veins. The CHIVA method is based on the elimination of the shunts that are responsible for the varicose veins. The CHIVA method is based on the elimination of the shunts that are responsible for the varicose veins. The CHIVA method is based on the elimination of the shunts that are responsible for t

In [22]:
question = "Can you explain what is CHIVA Shunt Type 3 in just 100 words?"

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=150, temperature=0)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("\nModel Answer:")
print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Can you explain what is CHIVA Shunt Type 3 in just 100 words?

Model Answer:
Can you explain what is CHIVA Shunt Type 3 in just 100 words? CHIVA is a method of treating varicose veins by interrupting the recirculation of blood in the veins without interrupting the normal flow of blood. It is based on the principle that the blood flows from the periphery to the heart and from the upper part of the body to the lower part. In this method, the blood is not drained from the veins, but rather the recirculation is interrupted. This is done by identifying and interrupting the recirculation circuits, which are called shunts. There are different types of shunts, and the most common is shunt type 3. In shunt type 3, the blood flows from the deep veins to the superficial veins through a perforating vein, then to the saphenous vein, and then back to the deep veins through another perforating vein. In CHIVA, this shunt is interrupted by making a small incision in the saphenous vein to disc

In [24]:
clinical_case = """
CLINICAL CASE - ULTRASOUND ASSESSMENT:

A patient underwent duplex ultrasound examination of the right lower extremity with the following findings:

FLOW PATTERNS OBSERVED:
1. Clip 1 (SFJ-Knee): Antegrade (EP) flow detected from location N1 to N2 at the saphenofemoral junction level
2. Clip 2 (SFJ-Knee): Antegrade (EP) flow continues from N2 to N3 in the saphenous vein/perforating vein system
3. Clip 3 (SFJ-Knee): Retrograde (RP) flow detected from N3 back to N1 through the perforating vein system

FLOW ANALYSIS:
- Antegrade flow exists from N1 → N2 at the saphenofemoral junction
- Antegrade flow exists from N2 → N3 through a perforating vein
- Retrograde flow exists along N3, but NO retrograde flow along N2
- This creates a closed circuit: N1 → N2 → N3 → N1

CLINICAL QUESTION:
Based on this hemodynamic pattern, please:
1. Classify the shunt type (according to CHIVA classification)
2. Explain the pathophysiology
3. Propose a ligation strategy and follow-up plan
"""

question = f"{clinical_case}\n\nProvide a detailed response."

print(question)

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs,max_length=300,temperature=0)
diagnosis = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("="*60)
print("MODEL DIAGNOSIS:")
print("="*60)
print(diagnosis.replace(clinical_case, "").strip())

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CLINICAL CASE - ULTRASOUND ASSESSMENT:

A patient underwent duplex ultrasound examination of the right lower extremity with the following findings:

FLOW PATTERNS OBSERVED:
1. Clip 1 (SFJ-Knee): Antegrade (EP) flow detected from location N1 to N2 at the saphenofemoral junction level
2. Clip 2 (SFJ-Knee): Antegrade (EP) flow continues from N2 to N3 in the saphenous vein/perforating vein system
3. Clip 3 (SFJ-Knee): Retrograde (RP) flow detected from N3 back to N1 through the perforating vein system

FLOW ANALYSIS:
- Antegrade flow exists from N1 → N2 at the saphenofemoral junction
- Antegrade flow exists from N2 → N3 through a perforating vein
- Retrograde flow exists along N3, but NO retrograde flow along N2
- This creates a closed circuit: N1 → N2 → N3 → N1

CLINICAL QUESTION:
Based on this hemodynamic pattern, please:
1. Classify the shunt type (according to CHIVA classification)
2. Explain the pathophysiology
3. Propose a ligation strategy and follow-up plan


Provide a detailed re

In [25]:
clinical_case = """
CLINICAL CASE - ULTRASOUND ASSESSMENT:

A patient underwent duplex ultrasound examination of the right lower extremity with the following findings:

FLOW PATTERNS OBSERVED:
1. Clip 1 (SFJ-Knee): Antegrade (EP) flow detected from location N1 to N2 at the saphenofemoral junction level
2. Clip 2 (SFJ-Knee): Antegrade (EP) flow continues from N2 to N3 in the saphenous vein/perforating vein system
3. Clip 3 (SFJ-Knee): Retrograde (RP) flow detected from N3 back to N1 through the perforating vein system

FLOW ANALYSIS:
- Antegrade flow exists from N1 → N2 at the saphenofemoral junction
- Antegrade flow exists from N2 → N3 through a perforating vein
- Retrograde flow exists along N3, but NO retrograde flow along N2
- This creates a closed circuit: N1 → N2 → N3 → N1

CLINICAL QUESTION:
Based on this hemodynamic pattern, please:
1. Classify the shunt type (according to CHIVA classification)
2. Explain the pathophysiology
3. Propose a ligation strategy and follow-up plan
"""

question = f"{clinical_case}\n\nProvide a detailed response."

inputs = tokenizer(question, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_length=300,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.5,
    do_sample=True
)
diagnosis = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("="*60)
print("MODEL DIAGNOSIS:")
print("="*60)
print(diagnosis.replace(clinical_case, "").strip())

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL DIAGNOSIS:
Provide a detailed response. Your answer should include all relevant details regarding anatomy/physiologic mechanisms that underlie your proposed treatment recommendations.
ANSWER


In [28]:
# Simpler prompt format
case_summary = """
ULTRASOUND FINDINGS:
- Antegrade flow: N1 → N2 (at SFJ)
- Antegrade flow: N2 → N3 (perforating vein)
- Retrograde flow: N3 → N1 (perforating vein)
- NO retrograde flow at N2

What shunt type is this? Propose a ligation strategy.
Choose shunt type from Type 1 / Type 2A / Type 2B / Type 2C / Type 3 / Type 1+2 / No Shunt Detected
"""

inputs = tokenizer(case_summary, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_length=2500,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.5,
    do_sample=True
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True).replace(case_summary, "").strip()

print("CASE SUMMARY:")
print(case_summary)
print("\nMODEL RESPONSE:")
print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=2500) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CASE SUMMARY:

ULTRASOUND FINDINGS:
- Antegrade flow: N1 → N2 (at SFJ)
- Antegrade flow: N2 → N3 (perforating vein)
- Retrograde flow: N3 → N1 (perforating vein)
- NO retrograde flow at N2

What shunt type is this? Propose a ligation strategy.
Choose shunt type from Type 1 / Type 2A / Type 2B / Type 2C / Type 3 / Type 1+2 / No Shunt Detected


MODEL RESPONSE:
The correct answer would be "Type 2" with the rationale provided above.


In [45]:
# Simpler prompt format
case_summary = """
ULTRASOUND FINDINGS:
- Antegrade flow: N1 → N2 (at SFJ)
- Antegrade flow: N2 → N3 (perforating vein)
- Retrograde flow: N3 → N1 (perforating vein)
- NO retrograde flow at N2

What CHIVA shunt type is this? 
Choose shunt type from Type 1 / Type 2A / Type 2B / Type 2C / Type 3 / Type 1+2 / No Shunt Detected
Propose a suitable ligation strategy for this shunt type.
"""

inputs = tokenizer(case_summary, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_length=2500,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.5,
    do_sample=True
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True).replace(case_summary, "").strip()

print("CASE SUMMARY:")
print(case_summary)
print("\nMODEL RESPONSE:")
print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=2500) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CASE SUMMARY:

ULTRASOUND FINDINGS:
- Antegrade flow: N1 → N2 (at SFJ)
- Antegrade flow: N2 → N3 (perforating vein)
- Retrograde flow: N3 → N1 (perforating vein)
- NO retrograde flow at N2

What CHIVA shunt type is this? 
Choose shunt type from Type 1 / Type 2A / Type 2B / Type 2C / Type 3 / Type 1+2 / No Shunt Detected
Propose a suitable ligation strategy for this shunt type.


MODEL RESPONSE:
Answer:

Assistant: This case presents with an anterograde draining perforator, which can be seen as the red arrow pointing towards and then through valve V4. The blue arrows represent reflux into that same vessel via another "N" junction in addition to its own drainage point.

This finding corresponds most closely to **Type IIb** on classification by Hach et al., where there are two independent escape points; one proximal ("primary") located within or near of saphenofemoral region [V5]—which we have identified—and also distally ([Figure A]), but these do not share common connections between the

In [47]:
prompt_template = """
You are a CHIVA ultrasound diagnostic system. Classify shunt types EXACTLY as shown.

EXAMPLE 1:
Antegrade N1→N2, Retrograde N2→N1, NO other flows
→ Answer: Type 1

EXAMPLE 2:
Antegrade N1→N2→N3, Retrograde N3→N1, NO retrograde N2
→ Answer: Type 3

NOW CLASSIFY THIS CASE:
{reasoning}

INSTRUCTIONS:
1. Only answer with ONE shunt type from: Type 1 / Type 2A / Type 2B / Type 2C / Type 3 / Type 1+2 / No Shunt
2. Then propose ligation strategy
3. NO explanations, NO hallucinations
"""

case_reasoning = "- Antegrade N1→N2 at SFJ\n- Antegrade N2→N3 through perforating vein\n- Retrograde N3→N1\n- NO retrograde N2"

inputs = tokenizer(prompt_template.format(reasoning=case_reasoning), return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=200, temperature=0, repetition_penalty=2.0)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a CHIVA ultrasound diagnostic system. Classify shunt types EXACTLY as shown.

EXAMPLE 1:
Antegrade N1→N2, Retrograde N2→N1, NO other flows
→ Answer: Type 1

EXAMPLE 2:
Antegrade N1→N2→N3, Retrograde N3→N1, NO retrograde N2
→ Answer: Type 3

NOW CLASSIFY THIS CASE:
- Antegrade N1→N2 at SFJ
- Antegrade N2→N3 through perforating vein
- Retrograde N3→N1
- NO retrograde N2

INSTRUCTIONS:
1. Only answer with ONE shunt type from: Type 1 / Type 2A / Type 2B / Type 2C / Type 3 / Type 1+2 / No Shunt
2. Then propose ligation strategy
3. NO explanations, NO hallucinations
4.
5


In [48]:
prompt = """You are a CHIVA ultrasound diagnostic system. Classify shunt types EXACTLY as shown.

EXAMPLE 1:
Antegrade N1→N2, Retrograde N2→N1, NO other flows
→ Answer: Type 1

EXAMPLE 2:
Antegrade N1→N2→N3, Retrograde N3→N1, NO retrograde N2
→ Answer: Type 3

NOW CLASSIFY THIS CASE:
- Antegrade N1→N2 at SFJ
- Antegrade N2→N3 through perforating vein
- Retrograde N3→N1
- NO retrograde N2

ANSWER (Type only, then ligation strategy):"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=100,  # Use max_new_tokens, not max_length
    temperature=0,
    repetition_penalty=2.0,
    do_sample=False
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer_only = response.replace(prompt, "").strip()

print("="*60)
print("MODEL RESPONSE:")
print("="*60)
print(answer_only)


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


MODEL RESPONSE:
To classify this case using the given criteria for determining which veins to ligate in order of priority:

Answer : T4


In [8]:
test_questions = [
    # BASIC KNOWLEDGE
    ("What is CHIVA and what does it stand for?", "general"),
    ("Explain post-thrombotic syndrome (PTS) in detail", "general"),
    ("What is the CEAP classification and its components?", "general"),
    
    # SHUNT TYPES
    ("Describe CHIVA Shunt Type 1 with flow pattern", "shunt"),
    ("What are the key differences between Shunt Type 2A, 2B, 2C?", "shunt"),
    ("Explain Shunt Type 3: blood flow pattern and why it occurs", "shunt"),
    
    # TREATMENT/LIGATION
    ("What is the treatment approach for Type 1 shunts?", "treatment"),
    ("How do you manage Shunt Type 3? What do you ligate?", "treatment"),
    
    # ULTRASOUND/DIAGNOSIS
    ("What ultrasound findings indicate reflux?", "diagnosis"),
    ("How do you differentiate antegrade from retrograde flow?", "diagnosis"),
    
    # LYMPHATIC
    ("What is lymphedema and what causes it?", "lymphatic"),
    ("Describe chylous effusions", "lymphatic"),
]

results = {}

print("="*80)
print("TESTING MODEL KNOWLEDGE ON 12 QUESTIONS")
print("="*80)

for i, (question, category) in enumerate(test_questions, 1):
    print(f"\n[{i}/12] [{category.upper()}] {question}")
    print("-" * 80)
    
    inputs = tokenizer(question, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.5,
        do_sample=True
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.replace(question, "").strip()
    
    print(f"ANSWER:\n{answer}\n")
    
    # Track results by category
    if category not in results:
        results[category] = []
    results[category].append({
        "question": question,
        "answer": answer,
        "length": len(answer)
    })

# Summary
print("\n" + "="*80)
print("SUMMARY BY CATEGORY")
print("="*80)
for category in ["general", "shunt", "treatment", "diagnosis", "lymphatic"]:
    if category in results:
        count = len(results[category])
        avg_length = sum(r["length"] for r in results[category]) / count
        print(f"\n{category.upper()}: {count} questions")
        print(f"  Avg answer length: {avg_length:.0f} characters")

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


TESTING MODEL KNOWLEDGE ON 12 QUESTIONS

[1/12] [GENERAL] What is CHIVA and what does it stand for?
--------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


ANSWER:
How can I learn about this treatment?
CHIVAs stands by Compression, Heparin Treatment with Intra-arterial thrombolysis. It's a non-surgical method of treating varicose veins.
It was created in France at the end 1980s as an alternative to stripping or phlebectomy when doing surgery on large saphenous vein networks that were often found among patients suffering from advanced venous insufficiency (varicosities). The technique has been modified since then but its basic principles remain intact: compression applied first followed shortly afterwards—usually within minutes—with intravenous heparine injection into all affected superficial axial vessels; intra arterial infusion/infusion pumps are also used sometimes depending upon surgeon’s preference . Then localised high pressure ultrasound guided catheter directed fibrinoaseolizationof diseased segmentsfollowedbycompressionbandagingforawhileuntiltheveinshealupandbecomeuselessasadistributionsystemforsystem


[2/12] [GENERAL] Explain p

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

ANSWER:
A: Flow in the common hepatic duct against inspiration and with reversed flow on Valsalva maneuver B: Dilated intrahepatica Cervical portion of extra-hepatic bile D The correct answer is (A). Reflux from either or both liver cells, through a biliary leak point into peritoneum via an abnormal connection such as Rokitanski's cyst. It may be associated to cholangitis secondary infection 1) Intracavitary echoes due dilatation & hydrops around gall bladder wall2 ) Echogenic band across left lateral lobe3.) "Lumpy" echo pattern throughout its thickness4 )"Mushy"-appearing echolucent mass occupying most part of caust cavity5)."Mass-like"intra-uterine growth seen over fundus6 ). Acoustic shadowing indicating presence fo gas7.). Echo-poor material representing edema fluid8 . High velocity turbulent Doppler signals within bilila tree9"). Hypoecho


[10/12] [DIAGNOSIS] How do you differentiate antegrade from retrograde flow?
----------------------------------------------------------------

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


ANSWER:
How is the velocity of blood measured in a vessel?
Anterior and posterior
Blood Pressure: What are systolic, diastolic pressure; normal values for an adult at rest. Normal Blood Flow:
- Anterograde or forward direction (from atrium to ventricle).
1) Intrathoracic veins carry venous return up into heart.
2 ) When air enters pulmonary circulation through lungs , it forces out carbon dioxide gas .
3). The action potential propagates along myelinated fibers as fast impulse but unmyelinated axons via chemical transmission 4 ). A typical cardiac cycle lasts about half minute .


[11/12] [LYMPHATIC] What is lymphedema and what causes it?
--------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


ANSWER:
Lymphatic system refers to the network of vessels, nodes (glands), filters or ducts that carry a clear fluid called "lymph" through tissues. The purpose behind this circulatory pathway in our bodies are several: • To transport excess fluids from interstitial spaces back into circulation for re-use by blood cells; 2
• It serves as an important immune defense mechanism against bacteria/viruses via its function at filtering out microorganisms/ particles within tissue.
Lack Of Proper Function Causing Build Up In Tissues This occurs when there’s blockage somewhere along these pathways which prevents proper drainage leading excessive accumulation ("swelling")of liquid around affected areas such arms , legs etc…


[12/12] [LYMPHATIC] Describe chylous effusions
--------------------------------------------------------------------------------
ANSWER:
, the most severe complication of a tubal pregnancy.
A: Chyle is milky in appearance because it contains large amounts to lipids. B : It ca

In [9]:
test_questions = [
    ("What is CHIVA and what does it stand for?", "general"), # answer
    ("Explain post-thrombotic syndrome (PTS) in detail", "general"), # 
    ("What is the CEAP classification and its components?", "general"),
    
    ("Describe CHIVA Shunt Type 1 with flow pattern", "shunt"),
    ("What are the key differences between Shunt Type 2A, 2B, 2C?", "shunt"),
    ("Explain Shunt Type 3: blood flow pattern and why it occurs", "shunt"),
    
    ("What is the treatment approach for Type 1 shunts?", "treatment"),
    ("How do you manage Shunt Type 3? What do you ligate?", "treatment"),
    
    ("What ultrasound findings indicate reflux?", "diagnosis"),
    ("How do you differentiate antegrade from retrograde flow?", "diagnosis"),
    
    ("What is lymphedema and what causes it?", "lymphatic"),
    ("Describe chylous effusions", "lymphatic"),
]

print("="*80)
print("RE-TESTING WITH DETERMINISTIC DECODING + CONTEXT ANCHORING")
print("="*80)

for i, (question, category) in enumerate(test_questions, 1):
    # Add category anchor to prevent domain mixing
    prompt = f"[MEDICAL TOPIC: Venous/Lymphatic Disease]\n\n{question}"
    
    print(f"\n[{i}/12] [{category.upper()}] {question}")
    print("-" * 80)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0,  # DETERMINISTIC
        repetition_penalty=2.0,
        do_sample=False  # NO SAMPLING
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.replace(prompt, "").strip()
    
    print(f"ANSWER:\n{answer}\n")

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


RE-TESTING WITH DETERMINISTIC DECODING + CONTEXT ANCHORING

[1/12] [GENERAL] What is CHIVA and what does it stand for?
--------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


ANSWER:
CHiva stands to "Cure Hemodynamic in Vascular Surgery" which means that the treatment of varicose veins should be based on hemodynamics. The goal being not only cosmetic improvement but also a functional cure, i.e., eliminating reflux from saphenofemoral junction (SFJ) or incompetent perforators.

How was this method developed?

The idea came about after observing patients with recurrent vein disease despite conventional surgery such as stripping procedures where there were still remaining superficial flow abnormalities at 3-5 years follow-up period when most recurrences occur due mainly either valve incompetence/insufficiency failure secondary venocongestion/hyperemia caused by residual/superfluous inflow vessels like GSV branches etc..or


[2/12] [GENERAL] Explain post-thrombotic syndrome (PTS) in detail
--------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

ANSWER:
(Select all that apply.)
A. Flow towards the heart in a vein with normal anatomy B Both A and C D E
Answer:
D

Which of these is not an example from everyday life where compression has been used to treat veno-vascular disease?
Socks for people who have had varicose veins removed.
Compression stockings worn by patients after surgery on their leg arteries or deep valves, as well some other vascular conditions such phlebectomy etc..
The use fo elastic bandages following treatment at Varithena clinics which are designed specifically using graduated pressure levels over 3 layers so they can be applied directly onto compressed foam sclerotherapy wounds without causing damage due t he high pressures involved during application .
Catheter ablation procedures


[10/12] [DIAGNOSIS] How do you differentiate antegrade from retrograde flow?
--------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


ANSWER:
Anterograde Flow is defined as the normal direction of blood or lymph movement. This means that in veins, it flows towards and through a valve; for example:

  • Blood flowing out (antero)of your heart into an artery
    Lymh moving away(antero )from its reservoir to begin circulation.
Venography demonstrates this by filling structures normally with contrast.

Retrogradely refers back toward another structure such at:
-Contrast leaking backwards down/into vein during venogram due too poor compression on injection site -Lysing thrombus travelling distally after PE


[11/12] [LYMPHATIC] What is lymphedema and what causes it?
--------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


ANSWER:
Lym- phocele refers to the accumulation of fluid in a lumen (cyst). A cystic hygroma, for example. The term "lymphangiectasia" describes dilatation or ectasias; this can be congenital but usually results from obstruction elsewhere that leads ultimately 1)to distention at some point along an obstructive segment(s), which may result also because valves are incompetent so blood/serum accumulates between them as well.

Lydeman RA et al., J Vasc Surg20(5):839–46


[12/12] [LYMPHATIC] Describe chylous effusions
--------------------------------------------------------------------------------
ANSWER:
and the pathophysiology behind them. Chyle is a milky, yellow fluid that results from lymphangiogenesis in response to injury or obstruction of one (or more) major channels; it contains high levels (>10%) triglycerides as well other lipids such cholesterol esters (~2%), proteins >3%, lactate dehydrogenase activity similar with serum (<5%); white blood cells are also present but at low numb